# ⚙️ 04 — Feature Engineering

**Objective**: Transform raw transactional data into customer-level features suitable for ML, and define the churn target variable.

**Why this step matters**: ML models need structured, numeric features at a consistent granularity. Raw data has one row per order-item — we need one row per customer with aggregated behavioral features.

**Key Decision — Churn Definition**: A customer is defined as "churned" if they have NOT made a purchase in the last 90 days (relative to the most recent date in the dataset). This is an industry-standard threshold for e-commerce.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.feature_engine import define_churn, build_customer_features, prepare_ml_dataset
from src.utils import set_plot_style, COLORS, plot_correlation_heatmap

set_plot_style()

# Load cleaned data
df = pd.read_csv('../data/master_cleaned.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows ✓")

## 1. Feature Engineering Pipeline

We create **12 customer-level features** from the raw transactional data:

| # | Feature | Description | Business Intuition |
|---|---------|-------------|-------------------|
| 1 | `frequency` | Number of orders | More orders → more engaged |
| 2 | `monetary` | Total spend (R$) | Higher value customers are less likely to churn |
| 3 | `recency_days` | Days since last purchase | Key churn predictor |
| 4 | `avg_order_value` | Average per order | Purchase pattern indicator |
| 5 | `avg_review_score` | Mean review score | Satisfaction indicator |
| 6 | `review_count` | Reviews submitted | Engagement indicator |
| 7 | `tenure_days` | Days between first & last purchase | Loyalty indicator |
| 8 | `avg_days_between_orders` | Purchase frequency rhythm | Regular → less churn |
| 9 | `avg_installments` | Average installments | Payment behavior |
| 10 | `payment_type_diversity` | # distinct payment methods | Flexibility indicator |
| 11 | `late_delivery_rate` | % orders delivered late | Bad experience → churn |
| 12 | `category_diversity` | # unique categories bought | Exploration behavior |

In [ ]:
# Run the complete pipeline
ml_dataset = prepare_ml_dataset(df, churn_threshold_days=90)
ml_dataset.head(10)

## 2. Feature Distributions

In [ ]:
# Visualize feature distributions split by churn label
feature_cols = ['frequency', 'monetary', 'recency_days', 'avg_order_value',
                'avg_review_score', 'tenure_days', 'avg_days_between_orders',
                'late_delivery_rate', 'category_diversity']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i, col in enumerate(feature_cols):
    ax = axes[i // 3][i % 3]
    for label, color in [(0, COLORS['success']), (1, COLORS['danger'])]:
        data = ml_dataset[ml_dataset['churned'] == label][col]
        ax.hist(data, bins=30, alpha=0.6, color=color, edgecolor='none',
                label='Active' if label == 0 else 'Churned', density=True)
    ax.set_title(col, fontweight='bold', fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Churn Status', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Correlation Analysis

In [ ]:
# Correlation heatmap of all features + target
feature_cols_all = ['frequency', 'monetary', 'recency_days', 'avg_order_value',
                    'avg_review_score', 'review_count', 'tenure_days',
                    'avg_days_between_orders', 'avg_installments',
                    'payment_type_diversity', 'late_delivery_rate',
                    'category_diversity', 'churned']

fig = plot_correlation_heatmap(ml_dataset[feature_cols_all],
                                title='Feature Correlation with Churn Target')
plt.show()

# Show correlation with target
print("\nCorrelation with Churn (sorted by absolute value):")
corr_with_target = ml_dataset[feature_cols_all].corr()['churned'].drop('churned').abs().sort_values(ascending=False)
for feat, corr in corr_with_target.items():
    direction = "+" if ml_dataset[feature_cols_all].corr()['churned'][feat] > 0 else "-"
    print(f"  {feat:30s}  {direction}{corr:.3f}")

## 4. Class Balance Check

In [ ]:
# Check class balance
churn_counts = ml_dataset['churned'].value_counts()
print(f"Active (0): {churn_counts.get(0, 0):,} ({churn_counts.get(0, 0)/len(ml_dataset)*100:.1f}%)")
print(f"Churned (1): {churn_counts.get(1, 0):,} ({churn_counts.get(1, 0)/len(ml_dataset)*100:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
colors = [COLORS['success'], COLORS['danger']]
ax.bar(['Active', 'Churned'], churn_counts.values, color=colors, edgecolor='none', width=0.5)
for i, v in enumerate(churn_counts.values):
    ax.text(i, v + len(ml_dataset)*0.01, f'{v:,}\n({v/len(ml_dataset)*100:.1f}%)',
            ha='center', fontweight='bold')
ax.set_title('Churn Class Distribution', fontweight='bold', fontsize=14)
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.show()

print(f"\nNote: We use 'class_weight=balanced' in models to handle any imbalance.")

## 5. Save ML Dataset

In [ ]:
# Save for model training
ml_dataset.to_csv('../data/ml_dataset.csv', index=False)
print(f"✓ Saved ML dataset: {ml_dataset.shape[0]:,} customers × {ml_dataset.shape[1]} columns")
print(f"  File: data/ml_dataset.csv")
print(f"  Features: {ml_dataset.shape[1] - 2} (excluding customer_id and target)")

## Summary

| Output | Value |
|--------|-------|
| Customers | ~8,500 |
| Features | 12 |
| Target | `churned` (binary: 0 = active, 1 = churned) |
| Churn threshold | 90 days without purchase |

**Key Insight**: Recency (`recency_days`) shows the strongest correlation with churn — customers who haven't bought recently are most likely to churn.

**Next Step**: In notebook 05, we'll train and compare ML models using this dataset.